# FutureTraffic Colab 실행

위에서부터 코드 셀 왼쪽의 **▶ 버튼**을 하나씩 누릅니다.  
처음에는 연결 확인을 위해 5,000단계만 학습합니다.


## 1. Google Drive 연결

실행 후 나타나는 **Connect to Google Drive** 버튼을 누르고, 계정을 선택한 다음 **계속**을 누릅니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. 데이터 압축파일 확인

Google Drive의 `내 드라이브/FutureTraffic/` 안에 `FutureTraffic_colab_data.zip`이 있어야 합니다.


In [ ]:
from pathlib import Path

DRIVE_PROJECT = Path('/content/drive/MyDrive/FutureTraffic')
DATA_ZIP = DRIVE_PROJECT / 'FutureTraffic_colab_data.zip'
CHECKPOINT_ROOT = DRIVE_PROJECT / 'checkpoints'
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
assert DATA_ZIP.exists(), f'파일을 찾지 못했습니다: {DATA_ZIP}'
print('데이터 확인:', DATA_ZIP)


## 3. GitHub 코드 받기

저장소가 없으면 복제하고, 이미 있으면 최신 코드로 갱신합니다.


In [ ]:
import os
import subprocess

def run_command(command, cwd=None):
    """Run a command and show its output live in the Colab cell."""
    environment = os.environ.copy()
    environment['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        command,
        cwd=cwd,
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    return ''.join(lines)

REPO = Path('/content/FutureTraffic')
if (REPO / '.git').exists():
    run_command(['git', '-C', str(REPO), 'pull', '--ff-only'])
else:
    run_command(
        ['git', 'clone', 'https://github.com/wkdus0608/FutureTraffic.git', str(REPO)]
    )
print('코드 준비 완료:', REPO)


## 4. 필요한 프로그램 설치

처음 실행할 때 몇 분 걸릴 수 있습니다. 마지막에 오류 없이 끝나면 정상입니다.


In [ ]:
run_command(
    ['python', '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')]
)
print('설치 완료')


## 5. SUMO 데이터 풀기

Drive의 압축파일을 Colab 임시 저장공간에 풉니다.


In [ ]:
from zipfile import ZipFile

with ZipFile(DATA_ZIP) as archive:
    archive.extractall(REPO)

SCENARIO = REPO / 'data/processed/20220810_0500'
required = [
    SCENARIO / 'network.net.xml',
    SCENARIO / 'traffic_20220810_0500.rou.xml',
    SCENARIO / 'scenario.sumocfg',
]
assert all(path.exists() for path in required), '압축파일 안의 SUMO 파일을 확인하세요.'
print('SUMO 데이터 준비 완료')
for path in required:
    print('-', path.name)


## 6. SUMO-RL 연결 확인

관측값 45개, 녹색 신호 4개, 황색신호 3초, 보상 함수 queue가 나오면 성공입니다.


In [ ]:
output = run_command(
    ['python', 'src/check_rl_environment.py', '--steps', '10'],
    cwd=REPO,
)
assert '관측값 개수: 45' in output
assert '선택 가능한 녹색 신호: 4개' in output
assert '고정신호 황색시간: [3]초' in output
assert '황색신호 시간: 3초' in output
assert '보상 함수: queue' in output
print('SUMO-RL 연결 확인 성공')


## 7. 새 DQN 시험 학습

처음에는 5,000단계로 전체 과정만 확인합니다. 모델은 10,000단계마다, replay buffer는 50,000단계마다 Drive에 저장됩니다.


In [ ]:
TRAINING_SEED = 42  # @param {type:"integer"}
TIMESTEPS = 5000  # @param {type:"integer"}

run_command(
    [
        'python', 'src/train_dqn.py',
        '--seed', str(TRAINING_SEED),
        '--timesteps', str(TIMESTEPS),
        '--checkpoint-dir', str(CHECKPOINT_ROOT),
    ],
    cwd=REPO,
)


## 8. 중단된 학습 이어서 실행

런타임이 끊겼을 때만 `RESUME = True`로 바꿉니다. 가장 최근 replay buffer와 같은 단계의 모델을 자동으로 선택합니다.


In [ ]:
RESUME = False  # @param {type:"boolean"}
ADDITIONAL_TIMESTEPS = 5000  # @param {type:"integer"}

if RESUME:
    seed_checkpoints = CHECKPOINT_ROOT / f'seed_{TRAINING_SEED}'
    replay_files = sorted(
        seed_checkpoints.glob('*_replay_buffer_*_steps.pkl'),
        key=lambda path: path.stat().st_mtime,
    )
    assert replay_files, '저장된 replay buffer 체크포인트가 없습니다.'
    resume_replay = replay_files[-1]
    resume_model = resume_replay.with_name(
        resume_replay.name.replace('_replay_buffer_', '_').replace('.pkl', '.zip')
    )
    assert resume_model.exists(), f'같은 단계의 모델이 없습니다: {resume_model}'
    run_command(
        [
            'python', 'src/train_dqn.py',
            '--seed', str(TRAINING_SEED),
            '--timesteps', str(ADDITIONAL_TIMESTEPS),
            '--checkpoint-dir', str(CHECKPOINT_ROOT),
            '--resume-model', str(resume_model),
            '--resume-replay-buffer', str(resume_replay),
        ],
        cwd=REPO,
    )
else:
    print('새 학습만 실행했습니다. 이어서 학습이 필요할 때 RESUME을 켜세요.')


## 9. 고정신호·무작위·DQN 평가

같은 차량과 seed로 세 방법을 비교합니다.


In [ ]:
MODEL = REPO / f'results/dqn_20220810_0500/seed_{TRAINING_SEED}/dqn_204820.zip'
assert MODEL.exists(), f'새 환경으로 학습한 모델이 없습니다: {MODEL}'
run_command(
    [
        'python', 'src/evaluate_policies.py',
        '--seeds', '42',
        '--models', str(MODEL),
    ],
    cwd=REPO,
)


## 10. 모델과 결과를 Drive에 저장

Colab 연결이 종료되어도 결과가 남도록 Drive로 복사합니다.


In [ ]:
from datetime import datetime
import shutil

run_name = datetime.now().strftime('%Y%m%d_%H%M%S')
destination = DRIVE_PROJECT / 'runs' / run_name
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(REPO / 'results', destination)
print('Drive 저장 완료:', destination)


## 11. 평가표 확인


In [ ]:
from IPython.display import Markdown, display

report = REPO / 'results/evaluation_20220810_0500/report.md'
display(Markdown(report.read_text(encoding='utf-8')))


## 12. 최종 5개 독립 모델 실험

시험 학습과 평가가 정상이고, 여러 날짜를 나눈 `splits.json`이 있을 때만 실행합니다. 학습 날짜로 5개 모델을 각각 100,000단계 학습하고, 사용하지 않은 테스트 날짜에서 비교합니다.


In [ ]:
RUN_FINAL_EXPERIMENT = False  # @param {type:"boolean"}
FINAL_TIMESTEPS = 100000  # @param {type:"integer"}
FINAL_SEEDS = [42, 43, 44, 45, 46]
SPLIT_FILE = REPO / 'data/processed/splits.json'
FINAL_MODEL_ROOT = REPO / 'results/dqn_final'
FINAL_EVALUATION_ROOT = REPO / 'results/evaluation_final'
FINAL_CHECKPOINT_ROOT = CHECKPOINT_ROOT / 'final'

if RUN_FINAL_EXPERIMENT:
    assert SPLIT_FILE.exists(), (
        '최종 실험에는 날짜 분할 파일이 필요합니다: ' + str(SPLIT_FILE)
    )
    run_command(
        [
            'python', 'src/train_dqn.py',
            '--seeds', *map(str, FINAL_SEEDS),
            '--timesteps', str(FINAL_TIMESTEPS),
            '--checkpoint-dir', str(FINAL_CHECKPOINT_ROOT),
            '--output-dir', str(FINAL_MODEL_ROOT),
            '--split-file', str(SPLIT_FILE),
            '--split', 'train',
        ],
        cwd=REPO,
    )
    final_models = [
        FINAL_MODEL_ROOT / f'seed_{seed}/dqn_204820.zip'
        for seed in FINAL_SEEDS
    ]
    run_command(
        [
            'python', 'src/evaluate_policies.py',
            '--seeds', *map(str, FINAL_SEEDS),
            '--models', *map(str, final_models),
            '--split-file', str(SPLIT_FILE),
            '--split', 'test',
            '--output-dir', str(FINAL_EVALUATION_ROOT),
        ],
        cwd=REPO,
    )
    final_report = FINAL_EVALUATION_ROOT / 'report.md'
    display(Markdown(final_report.read_text(encoding='utf-8')))
else:
    print('시험 결과를 확인한 뒤 RUN_FINAL_EXPERIMENT를 켜세요.')
